In [5]:
!pip install faiss-cpu

  Using cached faiss_cpu-1.13.0-cp39-abi3-macosx_14_0_arm64.whl.metadata (7.7 kB)
Using cached faiss_cpu-1.13.0-cp39-abi3-macosx_14_0_arm64.whl (3.4 MB)


In [ ]:
import os
import asyncio
import logging
import re
import pandas as pd
from typing import Optional, List, Dict

# Подключаем LightRAG
from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

# --------------------------------------------------------------------------------
# Параметры OpenRouter
# --------------------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "Ваш ключ")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Выбранная модель через OpenRouter
OPENROUTER_MODEL = "qwen/qwen3-4b-instruct"  # или другая модель, проверенная тобой

# --------------------------------------------------------------------------------
# Prompt-шаблоны
# --------------------------------------------------------------------------------
ULTRA_STRICT_PROMPT = """You are an Information Extraction Engine operating in STRICT MODE.
You MUST output ONLY valid LightRAG extraction lines.

CRITICAL RULES (MUST FOLLOW):
1. Output ONLY the allowed formats below.
2. NO markdown, NO punctuation outside fields, NO bullets, NO commentary.
3. NO introductory text, NO explanations, NO blank lines.
4. DO NOT output anything before the first entity or relation line.
5. DO NOT generate any lines that do not strictly match the required formats.

VALID OUTPUT FORMATS (ONLY these):

For ENTITIES (exactly 4 fields):
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>

For RELATIONS (exactly 5 fields):
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

FIELD RULES:
- You MUST NOT use "#", "|", "#|" inside fields.
- Do NOT leave any field empty.
- <entity_type> MUST be one of: person, organization, location, concept, object, event, role.
- <relation_type> MUST be a simple verb-like string (e.g., "uses", "creates", "belongs_to", "mentions").

TERMINATION:
After ALL extraction lines, output EXACTLY:
<|COMPLETE|>

If NOTHING is extractable, output ONLY:
<|COMPLETE|>

Begin extraction now.
"""

QA_PROMPT_SYSTEM = "You are a helpful assistant for answering questions based on given context."
SUMMARIZE_PROMPT_SYSTEM = "You are a summarizer. Summarize the user's input concisely."

# --------------------------------------------------------------------------------
# Функции взаимодействия с OpenRouter
# --------------------------------------------------------------------------------
import requests

async def openrouter_chat_completion(messages: List[Dict], model: str = OPENROUTER_MODEL,
                                      max_tokens: int = 256, temperature: float = 0.7,
                                      top_p: float = 0.9, **kwargs) -> Optional[str]:
    """
    Отправляет запрос в OpenRouter и возвращает ответ (строку).
    messages — список словарей с ключами "role" и "content"
    """
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        **kwargs
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    js = resp.json()
    # Пример ответа: js["choices"][0]["message"]["content"]
    try:
        return js["choices"][0]["message"]["content"]
    except Exception as e:
        logging.error("OpenRouter response parsing error: %s", e)
        return None

# --------------------------------------------------------------------------------
# Prompt-функция для LightRAG-LLM
# --------------------------------------------------------------------------------
async def llm_model_func(prompt: str, task: str = None, **kwargs) -> str:
    """
    Универсальная функция для LightRAG.
    task: "extract", "qa", "summarize", etc.
    """
    if task == "extract":
        system = ULTRA_STRICT_PROMPT
    elif task == "qa":
        system = QA_PROMPT_SYSTEM
    elif task == "summarize":
        system = SUMMARIZE_PROMPT_SYSTEM
    else:
        system = "You are an assistant."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    raw = await openrouter_chat_completion(messages=messages, **kwargs)
    if raw is None:
        return "" if task != "extract" else "<|COMPLETE|>"

    if task == "extract":
        return fix_extraction_output(raw)
    else:
        return raw.strip() + "\n<|COMPLETE|>"

# --------------------------------------------------------------------------------
# Пост-обработка extract-выхода
# --------------------------------------------------------------------------------
def fix_extraction_output(text: str) -> str:
    text = text.replace("<|COMPLETE|>", "")
    lines = text.splitlines()
    out = []
    entity_pattern = re.compile(r"^entity#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    relation_pattern = re.compile(r"^relation#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    for l in lines:
        l2 = l.strip()
        if entity_pattern.match(l2) or relation_pattern.match(l2):
            out.append(l2)
    out.append("<|COMPLETE|>")
    return "\n".join(out)

# --------------------------------------------------------------------------------
# Splitting / cleaning
# --------------------------------------------------------------------------------
def simple_split(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(length, start + chunk_size)
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def soft_clean(text: str) -> str:
    return text.strip().replace("\n", " ").replace("\r", " ")

# --------------------------------------------------------------------------------
# Инициализация LightRAG
# --------------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

embed_model_name = "BAAI/bge-m3"
# Модель эмбеддингов ["sentence-transformers/all-mpnet-base-v2"]
embed_model = SentenceTransformer(embed_model_name)

EMBED_DIM = 1024  # размерность модели

async def embedding_func(texts):
    return embed_model.encode(texts, convert_to_numpy=True)


def init_lightrag(work_dir: str = "rag_data") -> LightRAG:
    rag = LightRAG(
        working_dir=work_dir,
        llm_model_func=llm_model_func,
        # У тебя должен быть embedding_func и другие параметры, поменяй на свои
        embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, func=embedding_func),
        vector_storage="FaissVectorDBStorage",
        # Другие настройки…
    )
    return rag

# --------------------------------------------------------------------------------
# Загрузка датасета
# --------------------------------------------------------------------------------
async def load_dataset_into_lightrag(rag: LightRAG, df: pd.DataFrame,
                                     text_column: str = "text_clean"):
    batch = []
    for _, row in df.iterrows():
        text = row[text_column]
        if not isinstance(text, str):
            text = str(text)
        cleaned = soft_clean(text)
        chunks = simple_split(cleaned, chunk_size=200, overlap=50)
        batch.extend(chunks)
        if len(batch) >= 100:
            await rag.ainsert(batch)
            batch = []
    if batch:
        await rag.ainsert(batch)

# --------------------------------------------------------------------------------
# MAIN
# --------------------------------------------------------------------------------
async def main():
    # Пример данных
    df = pd.DataFrame({"text_clean": [
        "Apple Inc. is a company based in Cupertino. Tim Cook is the CEO.",
        "Gucci is a fashion brand. Its logo features a stylized G."
    ]})

    rag = init_lightrag()
    await rag.initialize_storages()
    await initialize_pipeline_status()
    await load_dataset_into_lightrag(rag, df)

    # Запрос
    query = "Какие организации и персоны упомянуты?"
    results = await rag.aquery(query, task="qa")  # или task="extract" в режиме извлечения
    print("=== RAG QUERY RESULT ===")
    print(results)

#if __name__ == "__main__":
#    asyncio.run(main())
await main()

INFO: [_] Created new empty graph file: rag_data/graph_chunk_entity_relation.graphml
INFO: Processing 2 document(s)
INFO: Extracting stage 1/2: unknown_source
INFO: Processing d-id: doc-5ec184135c74b0683ea256721cd39504
INFO: Extracting stage 2/2: unknown_source
INFO: Processing d-id: doc-f12d978ecb87351a4804d9ee3c4f197e
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
ERROR: LLM func: Error in decorated function for task 15876288960_807487.485096375: Object of type JsonKVStorage is not JSON serializable
ERROR: LLM func: Error in decorated function for task 15876288768_807487.486121041: Object of type JsonKVStorage is not JSON serializable
ERROR: Failed to extract entities and relationships: C[1/1]: chunk-5ec184135c74b0683ea256721cd39504: Object of type JsonKVStorage is not JSON serializable
ERROR: Failed to extract entities and relation

TypeError: LightRAG.aquery() got an unexpected keyword argument 'task'

In [ ]:
import os
import asyncio
import logging
import re
import pandas as pd
from typing import Optional, List, Dict

from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

# Параметры OpenRouter
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "Ваш ключ")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = "qwen/qwen3-4b-instruct"

# Prompt‑шаблоны
ULTRA_STRICT_PROMPT = """You are an Information Extraction Engine operating in STRICT MODE.
You MUST output ONLY valid LightRAG extraction lines.

CRITICAL RULES (MUST FOLLOW):
1. Output ONLY the allowed formats below.
2. NO markdown, NO punctuation outside fields, NO bullets, NO commentary.
3. NO introductory text, NO explanations, NO blank lines.
4. DO NOT output anything before the first entity or relation line.
5. DO NOT generate any lines that do not strictly match the required formats.

VALID OUTPUT FORMATS (ONLY these):

For ENTITIES (exactly 4 fields):
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>

For RELATIONS (exactly 5 fields):
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

FIELD RULES:
- You MUST NOT use "#", "|", "#|" inside fields.
- Do NOT leave any field empty.
- <entity_type> MUST be one of: person, organization, location, concept, object, event, role.
- <relation_type> MUST be a simple verb‑like string (e.g., "uses", "creates", "belongs_to", "mentions").

TERMINATION:
After ALL extraction lines, output EXACTLY:
<|COMPLETE|>

If NOTHING is extractable, output ONLY:
<|COMPLETE|>

Begin extraction now.
"""

QA_PROMPT_SYSTEM = "You are a helpful assistant for answering questions based on given context."
SUMMARIZE_PROMPT_SYSTEM = "You are a summarizer. Summarize the user's input concisely."

import requests

async def openrouter_chat_completion(
    messages: List[Dict],
    model: str = OPENROUTER_MODEL,
    max_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    **kwargs
) -> Optional[str]:
    # Фильтруем kwargs — оставляем только примитивные параметры, чтобы не передавать объекты типа JsonKVStorage
    allowed = {k: v for k, v in kwargs.items() if k in ("stop", "n", "model", "max_tokens", "temperature", "top_p")}
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        **allowed
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    js = resp.json()
    try:
        return js["choices"][0]["message"]["content"]
    except Exception as e:
        logging.error("OpenRouter response parsing error: %s", e)
        return None

async def llm_model_func(prompt: str, task: str = None, **kwargs) -> str:
    if task == "extract":
        system = ULTRA_STRICT_PROMPT
    elif task == "qa":
        system = QA_PROMPT_SYSTEM
    elif task == "summarize":
        system = SUMMARIZE_PROMPT_SYSTEM
    else:
        system = "You are an assistant."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    raw = await openrouter_chat_completion(messages=messages, **kwargs)
    if raw is None:
        return "<|COMPLETE|>" if task == "extract" else ""

    if task == "extract":
        return fix_extraction_output(raw)
    else:
        return raw.strip() + "\n<|COMPLETE|>"

def fix_extraction_output(text: str) -> str:
    text = text.replace("<|COMPLETE|>", "")
    lines = text.splitlines()
    out = []
    entity_pattern = re.compile(r"^entity#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    relation_pattern = re.compile(r"^relation#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$")
    for l in lines:
        l2 = l.strip()
        if entity_pattern.match(l2) or relation_pattern.match(l2):
            out.append(l2)
    out.append("<|COMPLETE|>")
    return "\n".join(out)

def simple_split(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(length, start + chunk_size)
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def soft_clean(text: str) -> str:
    return text.strip().replace("\n", " ").replace("\r", " ")

from sentence_transformers import SentenceTransformer

embed_model_name = "BAAI/bge-m3"
embed_model = SentenceTransformer(embed_model_name)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

async def embedding_func(texts):
    return embed_model.encode(texts, convert_to_numpy=True)

def init_lightrag(work_dir: str = "rag_data") -> LightRAG:
    rag = LightRAG(
        working_dir=work_dir,
        llm_model_func=llm_model_func,
        embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, func=embedding_func),
        vector_storage="FaissVectorDBStorage",
    )
    return rag

async def load_dataset_into_lightrag(rag: LightRAG, df: pd.DataFrame, text_column: str = "text_clean"):
    batch = []
    for _, row in df.iterrows():
        text = row[text_column]
        if not isinstance(text, str):
            text = str(text)
        cleaned = soft_clean(text)
        chunks = simple_split(cleaned, chunk_size=200, overlap=50)
        batch.extend(chunks)
        if len(batch) >= 100:
            await rag.ainsert(batch)
            batch = []
    if batch:
        await rag.ainsert(batch)

async def main():
    df = pd.DataFrame({"text_clean": [
        "Apple Inc. is a company based in Cupertino. Tim Cook is the CEO.",
        "Gucci is a fashion brand. Its logo features a stylized G."
    ]})

    rag = init_lightrag()
    await rag.initialize_storages()
    await initialize_pipeline_status()
    await load_dataset_into_lightrag(rag, df)

    query = "Какие организации и персоны упомянуты?"
    results = await rag.aquery(query, task="qa")
    print("=== RAG QUERY RESULT ===")
    print(results)

#if __name__ == "__main__":
#    asyncio.run(main())
await main()

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
INFO: [_] Created new empty graph file: rag_data/graph_chunk_entity_relation.graphml
INFO: Reset 2 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 2 document(s)
INFO: Extracting stage 1/2: unknown_source
INFO: Processing d-id: doc-5ec184135c74b0683ea256721cd39504
INFO: Extracting stage 2/2: unknown_source
INFO: Processing d-id: doc-f12d978ecb87351a4804d9ee3c4f197e
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
ERROR: LLM func: Error in decorated function for task 15888087936_807755.174985833: 400

TypeError: LightRAG.aquery() got an unexpected keyword argument 'task'